&nbsp;

## [ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) Result Check Positive Control (Heat stress-related datasets)

- [config_biosamples_hs_atacseq_hic_list.tsv](../../out/ENCODE_rE2G/config/config_biosamples_hs_atacseq_hic_list.tsv)で定義した14個のデータセットでの enhancer-gene interaction の相互作用の結果のうち、[Stress-induced transcriptional memory accelerates promoter-proximal pause release and decelerates termination over mitotic divisions](https://doi.org/10.1016/j.molcel.2021.03.007) の Data S2 に記載されている 遺伝子 のリストと整合しているかを調べる
- 

In [1]:
import datetime
import os
import pathlib
import subprocess

import polars as pl
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

## Check GSE154744 ~~Enhancer~~ (Gene) list

- [Data S2](https://ars.els-cdn.com/content/image/1-s2.0-S1097276521001751-mmc3.zip) からダウンロード
- 読み込んでみたところ、これは エンハンサーのリストではなく、エンハンサーに制御される 遺伝子のリストだと考えられる

In [2]:
gse154744_enhancer_regulated_gene_list = pl.read_csv(
    "../../Data/Database/GSE154744_enhancer_list/molcel_7891_mmc3.txt",
    separator="\t"
)

print(f"Number of RefSeq RNA ID: {gse154744_enhancer_regulated_gene_list.select("txID").n_unique()}")
print(f"Number of Gene Symbol: {gse154744_enhancer_regulated_gene_list.select("geneName").n_unique()}")
display(gse154744_enhancer_regulated_gene_list)

Number of RefSeq RNA ID: 227
Number of Gene Symbol: 227


chr,txStart,txEnd,strand,geneName,txID
str,i64,i64,str,str,str
"""chr18""",19230857,19284766,"""-""","""ABHD3""","""NM_138340"""
"""chr14""",77924372,77935815,"""+""","""AHSA1""","""NM_012111"""
"""chr2""",61404554,61414686,"""+""","""AHSA2""","""NM_152392"""
"""chr7""",36429411,36493401,"""+""","""ANLN""","""NM_001284302"""
"""chr9""",75766646,75785310,"""+""","""ANXA1""","""NM_000700"""
…,…,…,…,…,…
"""chr22""",22868060,22874624,"""-""","""ZNF280A""","""NM_080740"""
"""chr1""",23695463,23698330,"""+""","""ZNF436-AS1""","""NR_033690"""
"""chr19""",50529211,50552031,"""+""","""ZNF473""","""NM_015428"""


&nbsp;

&nbsp;

&nbsp;

## Target Gene Investigation

- 上記のファイルには、ありがたいことに、RefSeq RNA の ID がついていた
- これを元に、TogoID を使って、RefSeq RNA → Ensembl transcript -> Ensembl Gene -> HGNC -> HGNC Symbol を再取得する！

In [ ]:
# refseq_rna_id_list = gse154744_enhancer_regulated_gene_list.select(
#     "txID"
# ).unique().write_csv(
#     "../../out/ENCODE_rE2G/positive_control_gene_list/enhancer_regulated_gene_list_refseq_rna_id.txt",
#     include_header=False
# )

&nbsp;

&nbsp;

&nbsp;

## TogoID Script Execution (20261110)

- code: `bash ./scripts/togoid_convert.sh ./out/ENCODE_rE2G/positive_control_gene_list/enhancer_regulated_gene_list_refseq_rna_id.txt refseq_rna,ensembl_transcript,ensembl_gene,hgnc,hgnc_symbol ./out/ENCODE_rE2G/positive_control_gene_list/enhancer_regulated_gene_list_refseq_rna_togoid_convert_20261110.tsv`

&nbsp;

- relation: `RefSeq RNA` - is nearly equivalent to - `Ensembl transcript` - is transcribed from - `Ensembl gene` - is nearly equivalent to - `HGNC` - has synonym - `HGNC gene symbol`

In [3]:
togoid_convert_result = pl.read_csv(
    "../../out/ENCODE_rE2G/positive_control_gene_list/enhancer_regulated_gene_list_refseq_rna_togoid_convert_20261110.tsv",
    separator="\t"
).rename({
    "refseq_rna_id": "txID"
})

display(togoid_convert_result)

txID,ensembl_transcript_id,ensembl_gene_id,hgnc_id,hgnc_symbol_id
str,str,str,i64,str
"""NM_001040456""","""ENST00000006777""","""ENSG00000005486""",23082,"""RHBDD2"""
"""NM_001235""","""ENST00000358171""","""ENSG00000149257""",1546,"""SERPINH1"""
"""NM_017539""","""ENST00000261383""","""ENSG00000158486""",2949,"""DNAH3"""
"""NM_001144766""","""ENST00000426588""","""ENSG00000168259""",12392,"""DNAJC7"""
"""NR_110069""",null,null,null,null
…,…,…,…,…
"""NM_001300928""","""ENST00000230085""","""ENSG00000112335""",11174,"""SNX3"""
"""NM_004620""","""ENST00000526995""","""ENSG00000175104""",12036,"""TRAF6"""
"""NM_001184897""","""ENST00000396282""","""ENSG00000172943""",20672,"""PHF8"""


In [4]:
# Ensembl transcript が増えてしまっていたので、新たに対応表をダウンロード
# 2026-11-10 access
refseq_rna_ensembl_transcript_mapping = pl.read_csv(
    "../../out/ENCODE_rE2G/positive_control_gene_list/enhancer_regulated_gene_list_refseq_rna_togoid_convert_maneselect_20261110.tsv",
    separator="\t"
).filter(
    pl.col("RefSeq RNA") == pl.col("RefSeq RNA_duplicated_0")
)

pl.Config.set_tbl_rows(10)
display(refseq_rna_ensembl_transcript_mapping)

RefSeq RNA,Ensembl transcript,Transcript flags,RefSeq RNA_duplicated_0
str,str,str,str
"""NM_001040456""","""ENST00000006777""","""Ensembl canonical GENCODE Basic APPRIS P1 TSL 1 MANE Select""","""NM_001040456"""
"""NM_001235""","""ENST00000358171""","""Ensembl canonical GENCODE Basic TSL 1 APPRIS P1 MANE Select""","""NM_001235"""
"""NM_017539""","""ENST00000261383""","""GENCODE Basic TSL 1""","""NM_017539"""
"""NM_001144766""","""ENST00000426588""","""GENCODE Basic TSL 2""","""NM_001144766"""
"""NM_001142305""","""ENST00000319296""","""TSL 1 Ensembl canonical APPRIS P1 GENCODE Basic MANE Select""","""NM_001142305"""
…,…,…,…
"""NM_001300928""","""ENST00000230085""","""APPRIS P2 Ensembl canonical TSL 1 MANE Select GENCODE Basic""","""NM_001300928"""
"""NM_004620""","""ENST00000526995""","""TSL 1 GENCODE Basic MANE Select Ensembl canonical APPRIS P2""","""NM_004620"""
"""NM_001184897""","""ENST00000396282""","""TSL 5 GENCODE Basic""","""NM_001184897"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1-to-1 に対応付ける

- Data S2 にある RefSeq RNA ID と Ensembl transcript が 1-1 に対応づかないため、以下のような作業を行った

In [5]:
# TogoID links some RefSeq IDs to the same gene on haplotype patches.
# Each of those RefSeq IDs has one transcript on chr1-22, X, or Y, and MANE Select is that copy when it exists.
gtf_full = pathlib.Path("../../Data/Database/GENCODE_v49/gencode.v49.chr_patch_hapl_scaff.annotation.gtf")
tx_patterns = "".join(
    f'transcript_id "{tx_id}.\n'
    for tx_id in refseq_rna_ensembl_transcript_mapping["Ensembl transcript"].unique().to_list()
)

gtf_hits = subprocess.run(
    ["grep", "-F", "-f", "-", str(gtf_full)],
    input=tx_patterns,
    check=True,
    capture_output=True,
    text=True,
)

tx_seqname = (
    pl.DataFrame({"line": [line for line in gtf_hits.stdout.splitlines() if "\ttranscript\t" in line]})
    .with_columns(
        pl.col("line").str.split("\t").list.get(0).alias("seqname"),
        pl.col("line").str.extract(r'transcript_id "([^"]+)"', 1).str.replace(r"\.\d+$", "").alias("Ensembl transcript"),
    )
    .select("Ensembl transcript", "seqname")
    .unique()
)

In [6]:
# Keep the primary-assembly transcript. This is one row per RefSeq.
refseq_tx_primary = (
    refseq_rna_ensembl_transcript_mapping
    .join(tx_seqname, on="Ensembl transcript", how="inner")
    .filter(pl.col("seqname").str.contains(r"^chr(\d+|X|Y)$"))
)

# semi keeps the TogoID columns and drops the patch copies
togoid_convert_result_one_to_one = togoid_convert_result.join(
    refseq_tx_primary.select(
        pl.col("RefSeq RNA").alias("txID"),
        pl.col("Ensembl transcript").alias("ensembl_transcript_id"),
    ),
    on=["txID", "ensembl_transcript_id"],
    how="semi",
    maintain_order="left",
)

print(f"RefSeq in the paper: {gse154744_enhancer_regulated_gene_list['txID'].n_unique()}")

print(f"Ensembl Gene ID Count: {togoid_convert_result_one_to_one['ensembl_gene_id'].n_unique()}")
print(f"one-to-one: {togoid_convert_result_one_to_one.height}")
display(togoid_convert_result_one_to_one)

RefSeq in the paper: 227
Ensembl Gene ID Count: 222
one-to-one: 222


txID,ensembl_transcript_id,ensembl_gene_id,hgnc_id,hgnc_symbol_id
str,str,str,i64,str
"""NM_001040456""","""ENST00000006777""","""ENSG00000005486""",23082,"""RHBDD2"""
"""NM_001235""","""ENST00000358171""","""ENSG00000149257""",1546,"""SERPINH1"""
"""NM_017539""","""ENST00000261383""","""ENSG00000158486""",2949,"""DNAH3"""
"""NM_001144766""","""ENST00000426588""","""ENSG00000168259""",12392,"""DNAJC7"""
"""NM_001142305""","""ENST00000319296""","""ENSG00000179965""",29653,"""ZNF771"""
…,…,…,…,…
"""NM_001300928""","""ENST00000230085""","""ENSG00000112335""",11174,"""SNX3"""
"""NM_004620""","""ENST00000526995""","""ENSG00000175104""",12036,"""TRAF6"""
"""NM_001184897""","""ENST00000396282""","""ENSG00000172943""",20672,"""PHF8"""


In [7]:
gse154744_enhancer_regulated_gene_list_join = gse154744_enhancer_regulated_gene_list.join(
    togoid_convert_result_one_to_one,
    on="txID",
    how="inner"
).filter(
    pl.col("ensembl_transcript_id").is_not_null()
).rename({
    "ensembl_transcript_id": "transcript_id_base",
    "ensembl_gene_id": "gene_id_base",
    "hgnc_id": "hgnc_id_base"
    }
)

print(f"Number of converted genes: {gse154744_enhancer_regulated_gene_list_join.select("hgnc_symbol_id").n_unique()}")
display(gse154744_enhancer_regulated_gene_list_join)

Number of converted genes: 222


chr,txStart,txEnd,strand,geneName,txID,transcript_id_base,gene_id_base,hgnc_id_base,hgnc_symbol_id
str,i64,i64,str,str,str,str,str,i64,str
"""chr18""",19230857,19284766,"""-""","""ABHD3""","""NM_138340""","""ENST00000289119""","""ENSG00000158201""",18718,"""ABHD3"""
"""chr14""",77924372,77935815,"""+""","""AHSA1""","""NM_012111""","""ENST00000216479""","""ENSG00000100591""",1189,"""AHSA1"""
"""chr7""",36429411,36493401,"""+""","""ANLN""","""NM_001284302""","""ENST00000396068""","""ENSG00000011426""",14082,"""ANLN"""
"""chr9""",75766646,75785310,"""+""","""ANXA1""","""NM_000700""","""ENST00000257497""","""ENSG00000135046""",533,"""ANXA1"""
"""chr1""",114437370,114447525,"""-""","""AP4B1""","""NM_001253852""","""ENST00000369569""","""ENSG00000134262""",572,"""AP4B1"""
…,…,…,…,…,…,…,…,…,…
"""chr22""",22868060,22874624,"""-""","""ZNF280A""","""NM_080740""","""ENST00000302097""","""ENSG00000169548""",18597,"""ZNF280A"""
"""chr1""",23695463,23698330,"""+""","""ZNF436-AS1""","""NR_033690""","""ENST00000335648""","""ENSG00000249087""",25122,"""ZNF436-AS1"""
"""chr19""",50529211,50552031,"""+""","""ZNF473""","""NM_015428""","""ENST00000270617""","""ENSG00000142528""",23239,"""ZNF473"""


&nbsp;

&nbsp;

&nbsp;

## Join HN-score transcript list (as a target)

- ターゲット側の情報だけ、HN-score を算出した 全て の transcript から統合する
- まずはこちらに統合する！
- 相互作用が予測できているかどうかをまずは確認するため、こちらに統合する
- gene_id レベルでの統合を行う

In [8]:
gencode_v49_fantabio_enhancer_regulated_gene_list = pl.read_parquet(
    "../../out/fantabio/parquet_file/HNscore_transcript_all_fantabio_v261001.parquet"
).with_columns(
    pl.col("transcript_id").str.replace(r"\.\d+$", "").alias("transcript_id_base"),
    pl.col("gene_id").str.replace(r"\.\d+$", "").alias("gene_id_base"),
    pl.col("HGNC_ID").str.extract(r"(\d+)$", 1).cast(pl.Int64).alias("hgnc_id_base"),
).join(
    gse154744_enhancer_regulated_gene_list_join,
    on=["gene_id_base", "hgnc_id_base"], # transcript id を除いて、gene level で統合する
    how="inner"
).drop(
    "chr",
    "txStart",
    "txEnd",
    "strand_right",
    "geneName",
    "hgnc_symbol_id"
)

print(f"Number of HN-score transcript - enhancer-regulated gene list: {gencode_v49_fantabio_enhancer_regulated_gene_list.select("gene_id_base", "hgnc_id_base").n_unique()}")
display(gencode_v49_fantabio_enhancer_regulated_gene_list)

Number of HN-score transcript - enhancer-regulated gene list: 221


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,transcript_id_base,gene_id_base,hgnc_id_base,txID,transcript_id_base_right
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,str,str,i64,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red""","""ENST00000309758""","""ENSG00000173110""",5239,"""NM_002155""","""ENST00000309758"""
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28106""","""chr1""",161524161,"""161524546""","""cp4@HSPA6""","[""ENST00000309758.6"", ""NM_002155.5"", … ""AK295212""]","""0""",0,"[{""EH38E2843855"",""PLS""}]","[{""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX3322007"",560}]}, {""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX13551280"",511}]}, … {""MED12"",511,[{""SRX10852993"",511}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr1:161494328..161494332,+;hg_6264.1"",""p1@HSPA6""}]",null,0.943234,"""PLA""","""204,0,0""","""red""","""ENST00000309758""","""ENSG00000173110""",5239,"""NM_002155""","""ENST00000309758"""
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_456076""","""chr1""",161524872,"""161524930""","""FCHS_456076""","[""KJ897015""]","""-214""",-332,"[{""EH38E2843856"",""pELS""}, {""EH38E3981534"",""PLS""}]","[{""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX12008350"",1465}]}, {""ZNF687"",2211,[{""SRX10478566"",2211}]}, … {""NR5A2"",516,[{""SRX10478059"",516}]}]","[{""HS_00013404"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00013405"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,1.0,"""ELA""","""255,202,74""","""yellow""","""ENST00000309758""","""ENSG0000017

&nbsp;

&nbsp;

## ENCODE-rE2G 用に整形

- 既にフィルタリングしたデータを ENCODE-rE2G 用に統合！

In [9]:
# target promoter side transcript candidate
fantabio_HN2_gencode_v49_target_transcript_candidate = gencode_v49_fantabio_enhancer_regulated_gene_list.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "transcript_id",
    "txID",
    "HN2",
    "transcript_type",
    "gene_id",
    "gene_id_base",
    "gene_symbol",
    "HGNC_ID",
    "strand",
    "RNACentral_ID",
    "annotation_remarks",
    "tags"
).unique().rename(
    # assign frag name "t" to target promoter side
    {
        "cre_id": "target_cre_id",
        "cre_name": "target_cre_name",
        "fantabio_cre_class": "target_fantabio_cre_class",
        "transcript_id": "t_transcript_id",
        "txID": "t_txID",
        "HN2": "t_HN2",
        "transcript_type": "t_transcript_type",
        "gene_id": "t_gene_id",
        "gene_id_base": "t_gene_id_base",
        "HGNC_ID": "t_HGNC_ID",
        "gene_symbol": "t_gene_symbol",
        "strand": "t_strand",
        "RNACentral_ID": "t_RNACentral_ID",
        "annotation_remarks": "t_annotation_remarks",
        "tags": "t_tags"
    }
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# Intergration of ENCODE-rE2G Result

In [10]:
encode_re2g_result = pl.read_parquet(
    "../../out/ENCODE_rE2G/parquet_file/encode_re2g_hs_join_v261113.parquet"
).filter(
    pl.col("isSelfPromoter") == False
)

print(f"encode_re2g_result count: {encode_re2g_result.select(["cre_id", "target_cre_id"]).n_unique()}")
pl.Config.set_tbl_rows(10)
display(encode_re2g_result)

encode_re2g_result count: 368398


ENCODE-rE2G.Score,re2g_dataset,cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,class,isSelfPromoter,distanceToTSS,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features,target_tss_info,target_cre_info,target_cre_overlap_info
f64,str,str,str,str,str,i64,f64,str,str,str,str,str,bool,f64,struct[3],struct[3],struct[8],struct[5],struct[10],struct[2]
0.194134,"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",358,0.854415,"""TRIM62""","""FCHS_10021""","""cp3@TRIM62""","""PLA""","""genic""",false,84675.0,"{""chr1"",33097145,33097645}","{33097084,33097503,0.614679}","{33182070,""ENSG00000116525"",0.524194,null,0.013442,13,1,506.041996}","{""chr1"",33181820,33182320,0,""-""}","{""chr1"",33181859,33182582,0,""."",33182047,33182188,""0,0,204"",""directionality:-0.999247|class:PLA"",-0.999247}","{461,0.922}"
0.342709,"""GSE154744_K562_42C_25min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",353,0.842482,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15223.0,"{""chr1"",33097150,33097650}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.383964,null,0.044833,2,1,220.597881}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.320048,"""GSE154744_K562_42C_60min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",288,0.687351,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15288.0,"{""chr1"",33097215,33097715}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.402521,null,0.044762,4,1,253.0904}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.312414,"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",358,0.854415,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15218.0,"{""chr1"",33097145,33097645}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.466515,null,0.044839,4,1,506.041996}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.291194,"""GSE173536_lymphblastoid_d21_42C_60min""","""FCHS_100026""","""FCHS_100026""","""chr12""","""ELA""",201,1.0,"""RASSF9""","""FCHS_100044""","""cp1@RASSF9""","""PLA""","""intergenic""",false,411922.0,"{""chr12"",85423911,85425325}","{85423957,85424158,0.5}","{85836540,""ENSG00000198774"",0.357834,null,0.00446,9,1,0.0}","{""chr12"",85836290,85836790,0,""-""}","{""chr12"",85836188,85836732,0,""."",85836213,85836732,""0,0,255"",""directionality:-1|class:PLA"",-1.0}","{442,0.884}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.21537,"""GSE101731_HFF_44C_60min""","""FCHS_99979""","""FCHS_99979""","""chr12""","""ELA""",396,0.938389,"""SLC6A15""","""FCHS_99974""","""cp9@SLC6A15""","""PLA""","""intergenic""",false,10163.0,"{""chr12"",84922742,84923242}","{84922716,84923138,-0.948148}","{84912829,""ENSG00000072041"",0.581099,null,0.077244,1,1,0.0}","{""chr12"",84912579,84913079,0,""-""}","{""chr12"",84912565,84913337,0,""."",84912757,84912903,""0,0,204"",""directionality:-0.99533|class:PLA"",-0.99533}","{500,1.0}"
0.311435,"""GSE101731_HFF_44C_120min""","""FCHS_99987""","""cp1@TSPAN19""","""chr12""","""PLA""",423,1.0,"""SLC6A15""","""FCHS_99974""","""cp9@SLC6A15""","""PLA""","""genic""",false,103772.0,"{""chr12"",85016351,85016851}","{85016374,85016797,-0.977465}","{84912829,""ENSG00000072041"",0.486403,null,0.014404,3,1,0.0}","{""chr12"",84912579,84913079,0,""-""}","{""chr12"",84912565,84913337,0,""."",84912757,84912903,""0,0,204"",""directionality:-0.99533|class:PLA"",-0.99533}","{500,1.0}"
0.291244,"""GSE101731_HFF_44C_60min""","""FCHS_99987""","""cp1@TSPAN19"

In [12]:
e2g_join_unique_count = encode_re2g_result.group_by(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class"
).agg(
    pl.col("re2g_dataset").unique().alias("re2g_dataset"),
    pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").alias("enhancer_cre_overlap_frac_of_fantabio_cre"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").min().alias("enhancer_cre_overlap_frac_of_fantabio_cre_min"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").max().alias("enhancer_cre_overlap_frac_of_fantabio_cre_max"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").mean().alias("enhancer_cre_overlap_frac_of_fantabio_cre_mean"), # 該当するデータセットで同定された enhancer と fantabio の CRE の重複率の平均値

    pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
    pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
    pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"), # 該当するデータセットで同定された enhancer と target promoter の相互作用のスコアの平均値

    # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
    # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
    # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
    # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
).sort(
    by=["re2g_dataset_count", "ENCODE-rE2G.Score_mean"],
    descending=True
)

print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): {e2g_join_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level) (with all dataset): {e2g_join_unique_count.filter(pl.col("re2g_dataset_count") == e2g_join_unique_count["re2g_dataset_count"].max()).select(['cre_id', 'target_cre_id']).n_unique()}")
display(e2g_join_unique_count)

ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): 368398
ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level) (with all dataset): 977


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64
"""FCHS_272143""","""FCHS_272143""","""PLA""","""FCHS_272148""","""cp1@MTFP1""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min"", ""GSE173536_lymphblastoid_d21_42C_60min"", … ""GSE154744_K562_42C_30min_intact_hic""]",14,"[0.752244, 0.748654, … 0.906643]",0.560144,0.906643,0.767248,"[0.995691, 0.995606, … 0.971064]",0.971064,0.995691,0.988938
"""FCHS_272142""","""FCHS_272142""","""PLA""","""FCHS_272148""","""cp1@MTFP1""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min"", ""GSE173536_lymphblastoid_d21_42C_60min"", … ""GSE154744_K562_42C_30min_intact_hic""]",14,"[1.0, 1.0, … 1.0]",0.703704,1.0,0.961698,"[0.995691, 0.995606, … 0.971064]",0.971064,0.995691,0.988938
"""FCHS_208523""","""cp2@PLAUR""","""PLA""","""FCHS_208525""","""cp5@PLAUR""","""PLA""","[""GSE154744_K562_42C_60min"", ""GSE173536_lymphblastoid_d21_42C_60min"", … ""GSE154744_K562_42C_30min_intact_hic""]",14,"[0.900362, 0.727355, … 0.930254]",0.727355,0.946558,0.849249,"[0.995005, 0.994978, … 0.941883]",0.941883,0.995005,0.988152
"""FCHS_168111""","""cp6@NCOR1""","""PLA""","""FCHS_168114""","""cp4@PIGL""","""PLA""","[""GSE154744_K562_42C_40min"", ""GSE154744_K562_42C_60min"", … ""GSE105028_hESC_42C_60min""]",14,"[0.775947, 1.0, … 1.0]",0.775947,1.0,0.944928,"[0.99309, 0.992987, … 0.960316]",0.960316,0.99309,0.98727
"""FCHS_133635""","""FCHS_133635""","""PLA""","""FCHS_133639""","""cp3@OCA2""","""PLA""","[""GSE101731_HFF_44C_60min"", ""GSE209675_MCF7_42C_60min"", … ""GSE154744_K562_42C_30min_intact_hic""]",14,"[0.889186, 0.933244, … 0.667557]",0.660881,0.933244,0.79506,"[0.996407, 0.992461, … 0.953398]",0.953398,0.996407,0.986245
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_580787""","""FCHS_580787""","""ELA""","""FCHS_406577""","""cp1@PSKH2""","""ELA""","[""GSE209675_MCF7_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.179003],0.179003,0.179003,0.179003
"""FCHS_17956""","""FCHS_17956""","""ELA""","""FCHS_18071""","""cp1@CCN1,cp6@CCNA2""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.179002],0.179002,0.179002,0.179002
"""FCHS_17958""","""FCHS_17958""","""ELA""","""FCHS_18071""","""cp1@CCN1,cp6@CCNA2""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min""]",1,[0.85473],0.85473,0.85473,0.85473,[0.179002],0.179002,0.179002,0.179002


In [13]:
e2g_join_unique_count_join = e2g_join_unique_count.join(
    fantabio_HN2_gencode_v49_target_transcript_candidate,
    on=["target_cre_id", "target_cre_name", "target_fantabio_cre_class"],
    how="inner"
).sort(
    by=["t_HN2", "re2g_dataset_count"],
    descending=[True, True]
)

print(f"Number of up5p transcript - ENCODE-rE2G result: {e2g_join_unique_count_join.select("t_gene_id_base").n_unique()}")
display(e2g_join_unique_count_join)

Number of up5p transcript - ENCODE-rE2G result: 208


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,t_transcript_id,t_txID,t_HN2,t_transcript_type,t_gene_id,t_gene_id_base,t_gene_symbol,t_HGNC_ID,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,str,i64,str,str,str,str,str,str,str,list[str],list[str]
"""FCHS_28114""","""FCHS_28114""","""PLA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE154744_K562_42C_12_5min"", ""GSE101731_HFF_44C_60min"", … ""GSE105028_hESC_42C_60min""]",14,"[0.925816, 0.934718, … 0.888724]",0.734421,1.0,0.86011,"[0.962253, 0.956158, … 0.317242]",0.317242,0.962253,0.85706,"""ENST00000309758.6""","""NM_002155""",335,"""protein_coding""","""ENSG00000173110.8""","""ENSG00000173110""","""HSPA6""","""HGNC:5239""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
"""FCHS_28112""","""FCHS_28112""","""ELA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min"", ""GSE267931_HUVEC_39C_120min"", … ""GSE209675_MCF7_42C_60min""]",13,"[0.782114, 1.0, … 0.785366]",0.765854,1.0,0.867292,"[0.947753, 0.883595, … 0.281932]",0.281932,0.947753,0.722447,"""ENST00000309758.6""","""NM_002155""",335,"""protein_coding""","""ENSG00000173110.8""","""ENSG00000173110""","""HSPA6""","""HGNC:5239""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
"""FCHS_28117""","""FCHS_28117""","""ELA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE154744_K562_42C_30min"", ""GSE154744_K562_42C_40min"", … ""GSE280307_THP1_42C_120min""]",13,"[1.0, 1.0, … 1.0]",0.546939,1.0,0.905651,"[0.868326, 0.866603, … 0.362252]",0.362252,0.868326,0.694225,"""ENST00000309758.6""","""NM_002155""",335,"""protein_coding""","""ENSG00000173110.8""","""ENSG00000173110""","""HSPA6""","""HGNC:5239""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
"""FCHS_28083""","""FCHS_28083""","""PLA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE154744_K562_42C_30min_intact_hic"", ""GSE154744_K562_42C_60min"", … ""GSE173536_lymphblastoid_t21_42C_60min""]",8,"[1.0, 1.0, … 1.0]",0.906298,1.0,0.988287,"[0.462919, 0.401608, … 0.185239]",0.185239,0.462919,0.298355,"""ENST00000309758.6""","""NM_002155""",335,"""protein_coding""","""ENSG00000173110.8""","""ENSG00000173110""","""HSPA6""","""HGNC:5239""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
"""FCHS_28082""","""FCHS_28082""","""PLA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE154744_K562_42C_30min_intact_hic"", ""GSE154744_K562_42C_60min"", … ""GSE173536_lymphblastoid_t21_42C_60min""]",8,"[1.0, 1.0, … 0.571429]",0.571429,1.0,0.885417,"[0.462919, 0.401608, … 0.185239]",0.185239,0.462919,0.298355,"""ENST00000309758.6""","""NM_002155""",335,"""protein_coding""","""ENSG00000173110.8""","""ENSG00000173110""","""HSPA6""","""HGNC:5239""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_526182""","""cp2@BICD1P1""","""ELA""","""FCHS_241763""","""cp4@CLK1""","""PLA""","[""GSE267931_HUVEC_39C_120min"", ""GSE101731_HFF_44C_120min""]",2,"[1.0, 1.0]",1.0,1.0,1.0,"[0.523591, 0.455743]",0.455743,0.523591,0.489667,"""ENST00000473565.5""","""NM_001162407""",-202,"""retained_intron""","""ENSG00000013441.18""","""ENSG00000013441""","""CLK1""","""HGNC:2068""","""-""",null,null,[]
"""FCHS_526181""","""FCHS_526181""","""ELA""","""FCHS_241763""","""cp4@CLK1""","""PLA""","[""GSE173536_lymphblastoid_d21_42C_60min"", ""GSE173536_lymphblastoid_t21_42C_60min""]",2,"[0.868949, 0.844156]",0.844156,0.868949,0.8565

In [14]:
# check
TAX1BP1 = e2g_join_unique_count_join.filter(
    pl.col("t_gene_symbol") == "TAX1BP1"
)

display(TAX1BP1)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,t_transcript_id,t_txID,t_HN2,t_transcript_type,t_gene_id,t_gene_id_base,t_gene_symbol,t_HGNC_ID,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,str,i64,str,str,str,str,str,str,str,list[str],list[str]
"""FCHS_378093""","""cp8@TAX1BP1""","""ELA""","""FCHS_378089""","""cp4@TAX1BP1,cp3@TAX1BP1-AS1""","""PLA""","[""GSE154744_K562_42C_40min"", ""GSE154744_K562_42C_12_5min"", … ""GSE267931_HUVEC_39C_120min""]",13,"[1.0, 1.0, … 1.0]",1.0,1.0,1.0,"[0.708294, 0.568857, … 0.192598]",0.192598,0.708294,0.407919,"""ENST00000916540.1""","""NM_001079864""",117,"""protein_coding""","""ENSG00000106052.15""","""ENSG00000106052""","""TAX1BP1""","""HGNC:11575""","""+""",null,"[""LRGASP""]","[""basic"", ""TAGENE""]"
"""FCHS_378094""","""cp5@TAX1BP1""","""ELA""","""FCHS_378089""","""cp4@TAX1BP1,cp3@TAX1BP1-AS1""","""PLA""","[""GSE154744_K562_42C_40min"", ""GSE154744_K562_42C_12_5min"", … ""GSE267931_HUVEC_39C_120min""]",13,"[1.0, 1.0, … 1.0]",1.0,1.0,1.0,"[0.708294, 0.568857, … 0.192598]",0.192598,0.708294,0.407919,"""ENST00000916540.1""","""NM_001079864""",117,"""protein_coding""","""ENSG00000106052.15""","""ENSG00000106052""","""TAX1BP1""","""HGNC:11575""","""+""",null,"[""LRGASP""]","[""basic"", ""TAGENE""]"
"""FCHS_571903""","""FCHS_571903""","""ELA""","""FCHS_378089""","""cp4@TAX1BP1,cp3@TAX1BP1-AS1""","""PLA""","[""GSE280307_THP1_42C_120min"", ""GSE154744_K562_42C_40min"", … ""GSE154744_K562_42C_30min_intact_hic""]",8,"[1.0, 1.0, … 1.0]",0.862319,1.0,0.98279,"[0.625849, 0.463824, … 0.281288]",0.281288,0.625849,0.396492,"""ENST00000916540.1""","""NM_001079864""",117,"""protein_coding""","""ENSG00000106052.15""","""ENSG00000106052""","""TAX1BP1""","""HGNC:11575""","""+""",null,"[""LRGASP""]","[""basic"", ""TAGENE""]"
"""FCHS_378076""","""FCHS_378076""","""ELA""","""FCHS_378089""","""cp4@TAX1BP1,cp3@TAX1BP1-AS1""","""PLA""","[""GSE101731_HFF_44C_60min"", ""GSE101731_HFF_44C_120min""]",2,"[1.0, 1.0]",1.0,1.0,1.0,"[0.327383, 0.296978]",0.296978,0.327383,0.312181,"""ENST00000916540.1""","""NM_001079864""",117,"""protein_coding""","""ENSG00000106052.15""","""ENSG00000106052""","""TAX1BP1""","""HGNC:11575""","""+""",null,"[""LRGASP""]","[""basic"", ""TAGENE""]"
"""FCHS_378157""","""FCHS_378157""","""PLA""","""FCHS_378089""","""cp4@TAX1BP1,cp3@TAX1BP1-AS1""","""PLA""","[""GSE101731_HFF_44C_60min"", ""GSE101731_HFF_44C_120min""]",2,"[0.9152, 0.7872]",0.7872,0.9152,0.8512,"[0.219428, 0.200473]",0.200473,0.219428,0.20995,"""ENST00000916540.1""","""NM_001079864""",117,"""protein_coding""","""ENSG00000106052.15""","""ENSG00000106052""","""TAX1BP1""","""HGNC:11575""","""+""",null,"[""LRGASP""]","[""basic"", ""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_378155""","""FCHS_378155""","""ELA""","""FCHS_378089""","""cp4@TAX1BP1,cp3@TAX1BP1-AS1""","""PLA""","[""GSE101731_HFF_44C_60min""]",1,[0.705357],0.705357,0.705357,0.705357,[0.219428],0.219428,0.219428,0.219428,"""ENST00000916538.1""","""NM_001079864""",-31,"""protein_coding""","""ENSG00000106052.15""","""ENSG00000106052""","""TAX1BP1""","""HGNC:11575""","""+""",null,"[""LRGASP""]","[""basic"", ""TAGENE""]"
"""FCHS_378146""","""FCHS_378146""","""PLA""","""FCHS_378089""","""cp4@TAX1BP1,cp3@TAX1BP1-AS1""","""PLA""","[""GSE101731_HFF_44C_120min""]",1,[0.872699],0.872699,0.872699,0.872699,[0.209847],0.209847,0.209847,0.209847,"""ENST00000916538.1""","""NM_001079864""",-31,"""protein_coding""","""ENSG00000106052.15""","""ENSG00000106052""","""TAX1BP1""","""HGNC:11575""","""+""",null,"[""LRGASP""]","[""basic"", ""TA